# Solutions to Exercises 12: More SPARQL

1. Load in the ontology `teaching.rdf` and 

a) Write a query using the OWLReady2 search capability to find all entities to which the entity with attribute `program_title="MSc Artificial Intelligence` has a direct relation (in either direction). You will need to use the ontology to help you with this. Plot the resulting network.


A `Program` can be involved in the following relations:

* `offers_programme(School, Program)`
* `has_module(Program, Module)`
* `is_enrolled_on(Student,Program)`
* `is_directed_by(Program,Staff)`

The other relation present in the ontology is 

* `is_taught_by(module,staff)`

In [1]:
from owlready2 import *
onto = get_ontology("teaching.rdf").load()
program = onto.search_one(program_title="MSc Artificial Intelligence")
students = onto.search(is_enrolled_on = program)
director = program.is_directed_by
modules = program.has_module
school = onto.search(offers_programme = program)
print(students)
print(director)
print(modules)
print(school)

[teaching.stu01, teaching.stu02, teaching.stu03, teaching.stu04, teaching.stu05]
[teaching.sta01]
[teaching.mod01, teaching.mod02, teaching.mod03, teaching.mod04, teaching.mod05, teaching.mod06, teaching.mod07]
[teaching.sch01]


Form these into triples for visualisation

In [2]:
ntriples = []

for i in students:
    ntriples.append((i.name, 'is_enrolled_on', program.name))


for i in director:
    ntriples.append((i.name, 'is_directed_by', program.name))


for i in modules:
    ntriples.append((program.name, 'has_module', i.name))

for i in school:
    ntriples.append((i.name, 'offers_programme', program.name))



Now get relations between the nodes: the only relation that allows this is `is_taught_by`

In [3]:
for d in director:
    teaches = onto.search(is_taught_by=d)
    for i in teaches:
        if i in modules:
            ntriples.append((i.name, 'is_taught_by', d.name))

Now we can form the graph

In [4]:
from pyvis.network import Network
net = Network()
# Get the node names
nodenames = set()
for i in ntriples:
    nodenames.add(i[0])
    nodenames.add(i[2])

for n in nodenames:
    net.add_node(n)

for n in ntriples:
    net.add_edge(n[0], n[2], title=n[1])

net.toggle_physics(True)
net.repulsion()
net.show_buttons(filter_=['physics'])
net.save_graph("nx.html")


    b) Repeat part (a) using SPARQL.

In [5]:
outgoing = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT $program ?relation ?entity
    WHERE
    {
        ?program rdf:type ONTO:Program
        ?program ONTO:program_title ?title
        $program ?relation ?entity
        FILTER($title="MSc Artificial Intelligence")
        FILTER(STRSTARTS(STR(?relation), "http://www.dummy.info/new.owl#"))
        FILTER(STRSTARTS(STR(?entity), "http://www.dummy.info/new.owl#"))

    }
    """))

for i in outgoing:  
    print(i)

[teaching.pro01, teaching.has_module, teaching.mod01]
[teaching.pro01, teaching.has_module, teaching.mod02]
[teaching.pro01, teaching.has_module, teaching.mod03]
[teaching.pro01, teaching.has_module, teaching.mod04]
[teaching.pro01, teaching.has_module, teaching.mod05]
[teaching.pro01, teaching.has_module, teaching.mod06]
[teaching.pro01, teaching.has_module, teaching.mod07]
[teaching.pro01, teaching.is_directed_by, teaching.sta01]


In [6]:
incoming = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT $entity ?relation ?program
    WHERE
    {
        ?program rdf:type ONTO:Program
        ?program ONTO:program_title ?title
        $entity ?relation ?program
        FILTER($title="MSc Artificial Intelligence")
        FILTER(STRSTARTS(STR(?relation), "http://www.dummy.info/new.owl#"))
        FILTER(STRSTARTS(STR(?entity), "http://www.dummy.info/new.owl#"))

    }
    """))

for i in incoming:  
    print(i)

[teaching.sch01, teaching.offers_programme, teaching.pro01]
[teaching.stu01, teaching.is_enrolled_on, teaching.pro01]
[teaching.stu02, teaching.is_enrolled_on, teaching.pro01]
[teaching.stu03, teaching.is_enrolled_on, teaching.pro01]
[teaching.stu04, teaching.is_enrolled_on, teaching.pro01]
[teaching.stu05, teaching.is_enrolled_on, teaching.pro01]


Now get the cross-links. This is turns out is really hard to do. Here's an examplem of how to do it. Note that there are several permutations of this because of the different directions of the three relations involved.

In [7]:
outgoingx = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT $entity ?relation2 ?entity2
    WHERE
    {
        ?program rdf:type ONTO:Program
        ?program ONTO:program_title ?title
        $program ?relation ?entity
        $program ?relation2 ?entity2
        ?entity ?relation3 ?entity2
        FILTER($title="MSc Artificial Intelligence")
        FILTER(STRSTARTS(STR(?relation), "http://www.dummy.info/new.owl#"))
        FILTER(STRSTARTS(STR(?entity), "http://www.dummy.info/new.owl#"))
        FILTER(STRSTARTS(STR(?relation2), "http://www.dummy.info/new.owl#"))
        FILTER(STRSTARTS(STR(?entity2), "http://www.dummy.info/new.owl#"))
        FILTER(STRSTARTS(STR(?relation3), "http://www.dummy.info/new.owl#"))
    }
    """))

for i in outgoingx:  
    print(i)

[teaching.mod05, teaching.is_directed_by, teaching.sta01]


2. Load in the ontology `HALD.rdf`.

a) Write a query to extract the 1- and 2- neighbourhoods of the `Disease` entity "Prostatic Neoplasms" and plot the graph. You may use  SPARQL or OWLReady2's `search` function for this.


In [8]:
from owlready2 import *
onto = get_ontology("./HALD.rdf").load()

# First check that we can find the node we want:

node = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT $node
    WHERE
    {
        ?node rdf:type ONTO:Disease
        ?node ONTO:entity "Prostatic_Neoplasms"
    }
    """))


print(node)


[[HALD.Prostatic_Neoplasms]]


In [9]:
node[0][0].entity

['Prostatic_Neoplasms']

Fetch all outgoing links. Write a couple of functions to help with this

In [10]:
def get_incoming_links(entityname):
    query = f"""
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?source ?relation ?target
    WHERE
    {{
        ?target rdf:type ONTO:Disease
        ?target ONTO:entity "{entityname}"
        ?source ?relation ?target
        FILTER(STRSTARTS(STR(?relation), "http://www.dummy.info/new.owl#"))
        FILTER(STRSTARTS(STR(?source), "http://www.dummy.info/new.owl#"))
    }}
    """
    return list(default_world.sparql(query))

def get_outgoing_links(entityname):
    query = f"""
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?source ?relation ?target
    WHERE
    {{
        ?source rdf:type ONTO:Disease
        ?source ONTO:entity "{entityname}"
        ?source ?relation ?target
        FILTER(STRSTARTS(STR(?relation), "http://www.dummy.info/new.owl#"))
        FILTER(STRSTARTS(STR(?target), "http://www.dummy.info/new.owl#"))
    }}
    """
    return list(default_world.sparql(query))

def get_all_relations(entityname):
    return get_incoming_links(entityname) + get_outgoing_links(entityname)

In [11]:
entityname = "Prostatic_Neoplasms"

relations = get_all_relations(entityname)
print(len(relations))

636


In [12]:
for r in relations:
    print(r)

[HALD.Neoplasms, HALD.defined, HALD.Prostatic_Neoplasms]
[HALD.Prostatic_Hyperplasia, HALD.defined, HALD.Prostatic_Neoplasms]
[HALD.LEP, HALD.associate, HALD.Prostatic_Neoplasms]
[HALD.Neoplasms, HALD.increase, HALD.Prostatic_Neoplasms]
[HALD.Death, HALD.increase, HALD.Prostatic_Neoplasms]
[HALD.NPEPPS, HALD.increase, HALD.Prostatic_Neoplasms]
[HALD.CP, HALD.increase, HALD.Prostatic_Neoplasms]
[HALD.Neoplasms, HALD.include, HALD.Prostatic_Neoplasms]
[HALD.Death, HALD.include, HALD.Prostatic_Neoplasms]
[HALD.IGFBP2, HALD.include, HALD.Prostatic_Neoplasms]
[HALD.AR, HALD.include, HALD.Prostatic_Neoplasms]
[HALD.Prostatic_Diseases, HALD.include, HALD.Prostatic_Neoplasms]
[HALD.Precancerous_Conditions, HALD.include, HALD.Prostatic_Neoplasms]
[HALD.HBEGF, HALD.include, HALD.Prostatic_Neoplasms]
[HALD.HOXC13, HALD.include, HALD.Prostatic_Neoplasms]
[HALD.SATB1, HALD.include, HALD.Prostatic_Neoplasms]
[HALD.Paraplegia, HALD.include, HALD.Prostatic_Neoplasms]
[HALD.PCA3, HALD.include, HALD.Pro

Merge them and plot the network

In [13]:
import pyvis.network as network

def plot_network(triples):
    net = network.Network()
    # get the unique nodes
    nodes = set()
    for t in triples:
        nodes.add(t[0])
        nodes.add(t[2])

    for n in nodes:
        net.add_node(n.entity[0])

    for t in triples:
        net.add_edge(t[0].entity[0],t[2].entity[0],type=t[1].name)

    net.toggle_physics(True)
    net.repulsion()
    net.show_buttons(filter_=['physics'])
    net.save_graph("network.html")
    return None

In [14]:
plot_network(relations)

Now for each of the unique nodes in the 1-neighbourhood, we need to compute their 1-neighbourhood.

In [15]:
# get the unique nodes
uniquenodes = set()
for r in relations:
    uniquenodes.add(r[0])
    uniquenodes.add(r[2])

for node in uniquenodes:
    entityname = node.entity[0]
    r = get_all_relations(entityname)
    relations += r

print(len(relations))
    

73944


In [16]:
# Too many relationships here. Sample.

import random
randomedges = random.choices(relations,k=5000)

plot_network(randomedges)

b) By systematically growing size of the neighbourhood, find a pathway from "Prostatic Neoplasms" to "Prostatitis" (hint: based on the names, do you expect these to be near or far apart?)

In [45]:
# Check to make sure it's not already there:
for r in relations:
    if (r[0].entity[0] == "Prostatitis" or r[2].entity[0] == "Prostatitis"):
        print(r)


[HALD.Prostatitis, HALD.associated, HALD.Prostatic_Neoplasms]
[HALD.Prostatitis, HALD.differentiate, HALD.Prostatic_Neoplasms]
[HALD.Prostatitis, HALD.play, HALD.Prostatic_Neoplasms]
[HALD.Prostatitis, HALD.evaluated, HALD.Prostatic_Neoplasms]
[HALD.Prostatic_Neoplasms, HALD.include, HALD.Prostatitis]
[HALD.Prostatic_Neoplasms, HALD.associated, HALD.Prostatitis]
[HALD.Prostatic_Neoplasms, HALD.identify, HALD.Prostatitis]
[HALD.Prostatic_Neoplasms, HALD.reveal, HALD.Prostatitis]
[HALD.Inflammation, HALD.correlate, HALD.Prostatitis]
[HALD.Inflammation, HALD.resemble, HALD.Prostatitis]
[HALD.Urinary_Tract_Infections, HALD.associate, HALD.Prostatitis]
[HALD.Infections, HALD.include, HALD.Prostatitis]
[HALD.Hyperplasia, HALD.include, HALD.Prostatitis]
[HALD.Prostatic_Neoplasms, HALD.include, HALD.Prostatitis]
[HALD.Prostatic_Hyperplasia, HALD.include, HALD.Prostatitis]
[HALD.Prostatic_Intraepithelial_Neoplasia, HALD.include, HALD.Prostatitis]
[HALD.Death, HALD.associated, HALD.Prostatitis]


It is already there - there are multiple direct connections.